v1과 동일한 구조 그러나 텐스플로우 기반

## 구조설계
1. input:(H,W,C)  #H세로 픽셀수 W가로 픽셀수 C 채널수(색깔정보)
2. Block1: Con ->ReLU -> MaxPool #Con(이미지특징) ReLU(비선형) MaxPool (크기 줄이고 핵심정보 추출)
3. Block2: Con ->ReLU -> MaxPool
4. Block3: Con ->ReLU -> MaxPool
5. Head: GAP(or Flatten) → Dense → Output
6. Input shape
7. Output 방식 확정
8. Overfitting 방지

파일 가져오기

In [19]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
import os
folder = "/content/drive/MyDrive/DS팀 데이터 저장소/"
print(os.listdir(folder)[:50])

['ORAL CANCER DATASET', 'Oral Images Dataset', 'Oral Cancer Images for Classification', '데이터전처리_정현.ipynb', '원본데이터전처리_정현.ipynb', 'Baseline_CNN_v2.ipynb', '원본데이터전처리_정현.txt', 'Baseline_CNN_v1.ipynb', 'new_model.ipynb']


import 확인 GPU 체크

In [21]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

print("TF version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TF version: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


데이터 shape/dtype 체크

In [29]:
import numpy as np

X_train = np.load('X_train.npy')
y_train = np.load('y_train.npy')
X_val = np.load('X_val.npy')
y_val = np.load('y_val.npy')

print("X_train:", X_train.shape, X_train.dtype, "min/max:", X_train.min(), X_train.max())
print("y_train:", y_train.shape, y_train.dtype, "unique:", np.unique(y_train)[:20])

print("X_val:", X_val.shape, X_val.dtype)
print("y_val:", y_val.shape, y_val.dtype)

X_train: (1845, 224, 224, 3) float32 min/max: -2.117904 2.64
y_train: (1845,) int64 unique: [0 1]
X_val: (231, 224, 224, 3) float32
y_val: (231,) int64


In [23]:
print("X_train:", X_train.shape, X_train.dtype, "min/max:", X_train.min(), X_train.max())
print("y_train:", y_train.shape, y_train.dtype, "unique:", np.unique(y_train)[:20])

print("X_val:", X_val.shape, X_val.dtype)
print("y_val:", y_val.shape, y_val.dtype)

X_train: (1845, 224, 224, 3) float32 min/max: -2.117904 2.64
y_train: (1845,) int64 unique: [0 1]
X_val: (231, 224, 224, 3) float32
y_val: (231,) int64


입 출력 설정 세팅 체크

In [24]:
input_shape = X_train.shape[1:]  # (H, W, C)

unique = np.unique(y_train)
if len(unique) == 2 and set(unique.tolist()) <= {0, 1}:
    mode = "binary"
    num_classes = 1
    loss = keras.losses.BinaryCrossentropy()
    metrics = [keras.metrics.BinaryAccuracy(name="acc"), keras.metrics.AUC(name="auc")]
else:
    mode = "multiclass"
    num_classes = len(unique)
    # y가 정수 라벨(0..K-1)이라고 가정 (대부분 이거)
    loss = keras.losses.SparseCategoricalCrossentropy()
    metrics = [keras.metrics.SparseCategoricalAccuracy(name="acc")]

print("input_shape:", input_shape)
print("mode:", mode, "| num_classes:", num_classes)

input_shape: (224, 224, 3)
mode: binary | num_classes: 1


모델 체크
과적합 방지(구조는 v1과 같이 그대로)

In [25]:
def build_cnn(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)

    # Block 1: Conv -> ReLU -> MaxPool
    x = layers.Conv2D(32, (3,3), padding="same", use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(0.15)(x)

    # Block 2
    x = layers.Conv2D(64, (3,3), padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(0.20)(x)

    # Block 3
    x = layers.Conv2D(128, (3,3), padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(0.25)(x)

    # Head: GAP -> Dense -> Output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.40)(x)

    if num_classes == 1:
        outputs = layers.Dense(1, activation="sigmoid")(x)
    else:
        outputs = layers.Dense(num_classes, activation="softmax")(x)

    return keras.Model(inputs, outputs)

model = build_cnn(input_shape, num_classes)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 110,561 (431.88 KB)

 Trainable params: 110,113 (430.13 KB)

 Non-trainable params: 448 (1.75 KB)

컴파일

In [26]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=loss,
    metrics=metrics
)

콜백및 학습

In [27]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 18s 157ms/step - acc: 0.6435 - auc: 0.6599 - loss: 0.6567 - val_acc: 0.4199 - val_auc: 0.7420 - val_loss: 0.9841 - learning_rate: 0.0010
Epoch 2/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - acc: 0.6964 - auc: 0.7505 - loss: 0.5832 - val_acc: 0.4589 - val_auc: 0.7702 - val_loss: 0.7571 - learning_rate: 0.0010
Epoch 3/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - acc: 0.7146 - auc: 0.7767 - loss: 0.5634 - val_acc: 0.6710 - val_auc: 0.7733 - val_loss: 0.6139 - learning_rate: 0.0010
Epoch 4/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - acc: 0.6901 - auc: 0.7365 - loss: 0.5995 - val_acc: 0.4199 - val_auc: 0.7636 - val_loss: 1.1062 - learning_rate: 0.0010
Epoch 5/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - acc: 0.7227 - auc: 0.7822 - loss: 0.5490 - val_acc: 0.6407 - val_auc: 0.8259 - val_loss: 0.6057 - learning_rate: 0.0010
Epoch 6/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - acc: 0.7011 - auc: 0.7675 - loss: 0.5649 - val_acc: 0.7446 - val_auc: 0.8282 - v

검증셋 평가+예측

In [28]:
print("Eval:", model.evaluate(X_val, y_val, verbose=0))

pred = model.predict(X_val[:8])
print("pred shape:", pred.shape)

if num_classes == 1:
    print("pred (prob):", pred.reshape(-1))
else:
    print("pred class:", pred.argmax(axis=1))

Eval: [0.4125896096229553, 0.8441558480262756, 0.8893675804138184]
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
pred shape: (8, 1)
pred (prob): [0.06942859 0.9229203  0.9777991  0.42068487 0.8637687  0.88233477
 0.9932181  0.34001964]
